<b><font size="6" color="#E8800A">Feature work: encoding and scaling</font></b><br>

This notebook continues from nb2. It reads the cleaned training file and the decisions recorded in the cleaning log (imputation plan and `log1p` columns) and compares four ways of encoding the categorical columns (one-hot, ordinal, count and target encoding) with four scalers (none, standard, min-max and robust). Every combination is fitted inside each of the same 20 splits grouped by `customer_id`, so the validation rows never help fit an encoder, an imputer or a scaler.

In [8]:
import importlib.util
import subprocess
import sys
from pathlib import Path

if importlib.util.find_spec("category_encoders") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "category-encoders"]
    )

# `course_helpers.py` sits beside the notebook, and Python does not search
# that folder on its own.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Standard library: `Counter` tallies how often a column is selected,
# `inspect` reads a selector's signature, `textwrap` folds printed text.
import inspect
import textwrap
import warnings
from collections import Counter

# Arrays, frames and plots.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")

# Encoders for the categorical columns, and two association statistics
# (chi-square and point-biserial) for scoring a column against the target.
from category_encoders import CountEncoder, TargetEncoder
from scipy.stats import chi2_contingency, pointbiserialr

# Feature extraction first, then the selectors this notebook compares.
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_selection import (
    RFE,
    RFECV,
    SelectFromModel,
    SelectKBest,
    SequentialFeatureSelector,
    VarianceThreshold,
)

# The imputer that fills each split, the model every technique is scored
# through, the splits it is scored on, and the score reported for it.
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit, train_test_split

# The scalers and encoders refitted inside every training split, and the
# warning the solver raises when it stops at `max_iter`.
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
    StandardScaler,
)
from sklearn.exceptions import ConvergenceWarning

# The cleaning log written by nb2 and the plot colours (course_helpers.py).
from course_helpers import CleaningLog, PLOT_BLUE, PLOT_ORANGE

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (6, 5)

# Convergence, one-hot and pandas deprecation warnings are silenced once here.
# The source of each is known and expected.
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

The decisions inherited from nb2:

In [9]:
recipe_log = CleaningLog.load("../logs/train_cleaning_log.json")

print(f"{len(recipe_log)} decisions inherited from nb2\n")
for decision in recipe_log.steps:
    carried = ", ".join(sorted(decision.carries))
    print(f"  {decision.column:28s} {decision.action}")
    print(f"  {'':28s} carries -> {carried}\n")

10 decisions inherited from nb2

  purchase_date                converted from text to datetime ('%Y-%m-%d')
                               carries -> applied, dtype, format

  all feature columns          drop rows empty in every feature
                               carries -> applied

  prior_mean_purchase_value    recomputed as prior_total_spend / prior_purchase_count
                               carries -> applied

  prior_mean_gap_days          computed from customer_tenure_days and the purchase count
                               carries -> applied

  spend_365d                   inconsistent values blanked; filled from prior_total_spend or 0 where the window holds all of the history or none of it
                               carries -> applied

  prior_std_gap_days, prior_std_purchase_value 0 for one gap or one purchase
                               carries -> applied

  days_since_previous_purchase, spend_365d invalid values -> NaN ('blank the cell')
                   

`customer_id` is only used to group the splits and `purchase_date` is left out: every test purchase is later than the training purchases (nb1), so the model would have to extrapolate in time.

In [10]:
train = pd.read_csv("../data/interim/train_cleaned.csv",
                    index_col="ID", parse_dates=["purchase_date"])

X = train.drop(columns=["repeat_purchase_90d", "customer_id", "purchase_date"])
y = train["repeat_purchase_90d"]

splitter = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=RANDOM_STATE)
splits = list(splitter.split(X, y, groups=train["customer_id"]))

print(f"{X.shape[0]:,} rows x {X.shape[1]} source columns")
print(f"{X.isna().sum().sum():,} gaps in the file,"
      f" filled per split from that split's own training rows")
rates = [y.iloc[test_rows].mean() for _, test_rows in splits]
print(f"repeat rate {y.mean():.4f} overall, {min(rates):.4f}-{max(rates):.4f} per held-out set")
sizes = [len(test_rows) for _, test_rows in splits]
print(f"{len(splits)} 80/20 splits grouped by customer,"
      f" {min(sizes)}-{max(sizes)} rows held out in each")

6,469 rows x 35 source columns
6,276 gaps in the file, filled per split from that split's own training rows
repeat rate 0.5597 overall, 0.4265-0.6924 per held-out set
20 80/20 splits grouped by customer, 823-1710 rows held out in each


In [ ]:
fill_step = recipe_log.plan("feature columns")
numeric, categorical = fill_step["numeric"], fill_step["categorical"]
fill = fill_step["fill"]
log1p_columns = recipe_log.plan("log1p columns")["columns"]

# The imputer each logged fill names. Every split refits it on its training rows.
imputers = {"zero": SimpleImputer(strategy="constant", fill_value=0),
            "median": SimpleImputer(strategy="median"),
            "mode": SimpleImputer(strategy="most_frequent")}
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": {"numeric": imputers[fill["numeric"]],
                   "categorical": imputers[fill["categorical"]]},
          "log1p": log1p_columns}

print(f"{len(numeric)} numeric, {len(categorical)} categorical")
print(f"numeric gaps take the training {fill['numeric']},"
      f" categorical gaps the training {fill['categorical']}")
print(f"log1p on {len(log1p_columns)} of the {len(numeric)} numeric columns;"
      f" the other {len(numeric) - len(log1p_columns)} are"
      f" {[c for c in numeric if c not in log1p_columns]}")

29 numeric, 6 categorical
numeric gaps take the training KNN (k=5), categorical gaps the training mode
log1p on 26 of the 29 numeric columns; the other 3 are ['invoice_count_current_purchase', 'customer_tenure_days', 'random_noise']


The functions below prepare one split: fill the missing values, encode the categorical columns, apply `log1p` and scale. Each step is fitted on the training rows of the split only.

In [12]:
def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs, and any scikit-learn imputer
    works: each one is fitted on the training rows of its own columns and fills
    those columns in both halves, so the test rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, the gap of a
        # nullable column such as the booleans, so every gap becomes np.nan.
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        imputer.fit(blocks[0])
        train[columns], test[columns] = [imputer.transform(block) for block in blocks]
    return train, test


# One encoder and one scaler per candidate, refitted inside every split. One-hot
# always drops one level per column.
encoders = {
    "one-hot": OneHotEncoder(handle_unknown="ignore", drop="first",
                             sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value",
                              unknown_value=-1),
    "count": CountEncoder(normalize=True, handle_unknown=0,
                          handle_missing="value"),
    "target": TargetEncoder(handle_unknown="value", handle_missing="value"),
}
scalers = {"none": None, "standard": StandardScaler,
           "min-max": MinMaxScaler, "robust": RobustScaler}


def fold_matrices(train, test, y_train, recipe):
    """Return (A_train, A_test, names) for ONE split, every step fitted on `train`.

    The categorical block comes first in the matrix and the numeric block
    second. That width is not constant across splits, because a level absent
    from a training split contributes no column to it.
    """
    numeric, categorical = recipe["numeric"], recipe["categorical"]

    # 1. FILL first, because the spine ships with its gaps open.
    train, test = fill_missing(train, test,
                               [(numeric, recipe["fill"]["numeric"]),
                                (categorical, recipe["fill"]["categorical"])])

    # 2. ENCODE. Every encoder is handed the training outcome, and only target
    #    encoding reads it.
    encoder = encoders[recipe["encoding"]]
    cat_train = encoder.fit_transform(train[categorical], y_train)
    cat_test = encoder.transform(test[categorical])
    names = list(encoder.get_feature_names_out(categorical)) + numeric

    # 3. log1p on the logged columns. It fits nothing.
    num_train = train[numeric].to_numpy(dtype=float).copy()
    num_test = test[numeric].to_numpy(dtype=float).copy()
    where = [numeric.index(column) for column in recipe["log1p"]]
    num_train[:, where] = np.log1p(num_train[:, where])
    num_test[:, where] = np.log1p(num_test[:, where])

    # 4. SCALE, with the centre and spread of the training rows. Ordinal, count
    #    and target encoding turn a level into a number, so the scaler applies
    #    to them too; one-hot indicators stay 0/1.
    make = scalers[recipe["scaler"]]
    if make is not None:
        fitted = make().fit(num_train)
        num_train, num_test = fitted.transform(num_train), fitted.transform(num_test)
        if recipe["encoding"] != "one-hot":
            fitted = make().fit(cat_train)
            cat_train, cat_test = fitted.transform(cat_train), fitted.transform(cat_test)

    return (np.hstack([cat_train, num_train]),
            np.hstack([cat_test, num_test]),
            np.asarray(names))

In [13]:
def heldout_f1(recipe, frame=X, reduce=None, select=None):
    """Held-out F1 on every split, one row per split.

    `reduce(n)` builds a reducer of n components, refitted with the fewest
    components that keep 80% of the split's variance. `select(split)` builds a
    selector for that split. The model sees the columns `keep` marks: all of
    them, unless a selector keeps fewer.
    """
    rows = []
    for split, (train_index, test_index) in enumerate(splits):
        y_train = y.iloc[train_index]
        A_train, A_test, names = fold_matrices(
            frame.iloc[train_index], frame.iloc[test_index], y_train, recipe)
        if reduce is not None:
            probe = reduce(A_train.shape[1] - 1).fit(A_train)
            k = int(np.searchsorted(probe.explained_variance_ratio_.cumsum(), 0.80) + 1)
            reducer = reduce(k).fit(A_train)
            A_train, A_test = reducer.transform(A_train), reducer.transform(A_test)
            names = np.asarray([f"component {i}" for i in range(1, k + 1)])
        selector, keep = None, np.ones(len(names), dtype=bool)
        if select is not None:
            # The selector is handed the matrix WITH its column names, because a
            # filter that judges a categorical as a whole has to know which
            # dummies came from it.
            selector = select(split).fit(pd.DataFrame(A_train, columns=names),
                                         y_train)
            keep = selector.get_support()
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(A_train[:, keep], y_train)
        rows.append({"n": int(keep.sum()),
                     "F1": f1_score(y.iloc[test_index], model.predict(A_test[:, keep])),
                     "kept": list(names[keep]), "selector": selector})
    return pd.DataFrame(rows)

<b><font size="5" color="#E8800A">Encoder and scaler</font></b><br>

In [14]:
encoding_runs = {
    (encoding, scaler): heldout_f1({**recipe, "encoding": encoding, "scaler": scaler})
    for encoding in encoders for scaler in scalers
}
encoding_board = pd.DataFrame.from_dict(
    {pair: {"F1": run["F1"].mean(), "F1 SEM": run["F1"].sem()}
     for pair, run in encoding_runs.items()},
    orient="index",
).rename_axis(["encoding", "scaler"])

# idxmax returns the (encoding, scaler) label of the best row, not its score.
selected_encoding, selected_scaler = encoding_board["F1"].idxmax()
recipe = {**recipe, "encoding": selected_encoding, "scaler": selected_scaler}
# The selected recipe's own run: every comparison below is paired against it.
reference = encoding_runs[selected_encoding, selected_scaler]

print(encoding_board.round(4).to_string())
print(f"selected pair: {selected_encoding} + {selected_scaler}"
      " (highest held-out F1)")

                       F1  F1 SEM
encoding scaler                  
one-hot  none      0.7074  0.0140
         standard  0.7096  0.0142
         min-max   0.7091  0.0143
         robust    0.7097  0.0142
ordinal  none      0.7049  0.0146
         standard  0.7045  0.0144
         min-max   0.7071  0.0141
         robust    0.7042  0.0145
count    none      0.7094  0.0144
         standard  0.7070  0.0143
         min-max   0.7097  0.0144
         robust    0.7064  0.0143
target   none      0.7044  0.0146
         standard  0.7083  0.0141
         min-max   0.7101  0.0139
         robust    0.7071  0.0143
selected pair: target + min-max (highest held-out F1)


The sixteen combinations score between 0.7042 and 0.7101, and each score has a standard error of about 0.014, so no combination is really better than the others. The highest is target encoding with min-max scaling, but target encoding replaces each level with its repeat rate in the training rows: for `transport_method`, where many levels belong to a single customer, that amounts to memorising customers. One-hot encoding with min-max scaling scores almost the same (0.7091) without that risk, so it is the pair used in nb4.